## 1. Check GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime > Change runtime type > T4 GPU")

## 1.5. 🧹 Clean /content/ Directory (Fresh Start)

Run this cell to completely wipe everything and start fresh. **Use this if you changed accelerators or had issues.**

In [ ]:
import os
import shutil
import subprocess

print("🧹 COMPLETE /content/ DIRECTORY WIPE")
print("=" * 60)
print("⚠️  This will delete:")
print("  - All cloned repositories (Evo-1, LIBERO)")
print("  - All downloaded checkpoints")
print("  - All logs and cached files")
print("  - Everything in /content/ except Colab system files")
print("=" * 60)
print()

# Get list of items in /content/
content_items = []
if os.path.exists('/content'):
    content_items = [item for item in os.listdir('/content') 
                     if not item.startswith('.') and item != 'sample_data']

if content_items:
    print(f"📋 Found {len(content_items)} items to delete:")
    for item in content_items:
        item_path = os.path.join('/content', item)
        if os.path.isdir(item_path):
            # Get directory size
            result = subprocess.run(['du', '-sh', item_path], 
                                  capture_output=True, text=True)
            size = result.stdout.split()[0] if result.stdout else '?'
            print(f"  📁 {item}/ ({size})")
        else:
            # Get file size
            size_bytes = os.path.getsize(item_path)
            size_mb = size_bytes / (1024 * 1024)
            print(f"  📄 {item} ({size_mb:.1f} MB)")
    
    print()
    print("🗑️  Deleting all items...")
    
    deleted_count = 0
    failed_items = []
    
    for item in content_items:
        item_path = os.path.join('/content', item)
        try:
            if os.path.isdir(item_path):
                shutil.rmtree(item_path)
                print(f"  ✅ Deleted: {item}/")
            else:
                os.remove(item_path)
                print(f"  ✅ Deleted: {item}")
            deleted_count += 1
        except Exception as e:
            print(f"  ❌ Failed to delete {item}: {e}")
            failed_items.append(item)
    
    print()
    print("=" * 60)
    print(f"✅ Cleanup complete!")
    print(f"   Deleted: {deleted_count}/{len(content_items)} items")
    
    if failed_items:
        print(f"   Failed: {len(failed_items)} items: {', '.join(failed_items)}")
    
    # Verify /content/ is clean
    remaining = [item for item in os.listdir('/content') 
                if not item.startswith('.') and item != 'sample_data']
    
    if remaining:
        print(f"\n⚠️  Warning: {len(remaining)} items still remain:")
        for item in remaining:
            print(f"     - {item}")
    else:
        print("\n🎉 /content/ directory is completely clean!")
    
    print("=" * 60)
else:
    print("✅ /content/ directory is already clean!")
    print("   Nothing to delete.")

print()
print("💡 Next step: Run cell 2 to install dependencies")

## 2. Install All Dependencies

In [ ]:
%%bash
set -e

echo "📦 Installing system dependencies..."
apt-get update -qq
apt-get install -y git wget > /dev/null

echo "📦 Installing Python packages..."
pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
pip install -q transformers==4.45.2 timm==1.0.9 accelerate==1.2.1 diffusers==0.31.0
pip install -q websockets pyyaml opencv-python pillow "numpy>=1.26.4,<1.27.0" fvcore
pip install -q einops thop cloudpickle easydict hydra-core
pip install -q huggingface_hub
pip install -q pyngrok

echo "✅ All dependencies installed!"

## 3. Clone Evo-1 and Download Checkpoint

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# Clone Evo-1
if not Path("/content/Evo-1").exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print("✅ Evo-1 cloned")
    
    # Install Evo-1 requirements
    print("📦 Installing Evo-1 requirements...")
    !pip install -q -r /content/Evo-1/requirements.txt
    
    # Reinstall torch to fix version conflicts
    print("📦 Fixing torch versions...")
    !pip install -q --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    print("✅ Evo-1 requirements installed")
else:
    print("✅ Evo-1 already exists")

# Download checkpoint
checkpoint_dir = "/content/Evo-1/checkpoints/libero"
os.makedirs(checkpoint_dir, exist_ok=True)

if not Path(f"{checkpoint_dir}/config.json").exists():
    print("📥 Downloading checkpoint (1.4GB)...")
    snapshot_download(
        repo_id="MINT-SJTU/Evo1_LIBERO",
        local_dir=checkpoint_dir,
        local_dir_use_symlinks=False
    )
    print("✅ Checkpoint downloaded")
else:
    print("✅ Checkpoint already exists")

## 4. Setup Evo-1 Server

In [ ]:
import json

# Update config to use CUDA
config_path = "/content/Evo-1/checkpoints/libero/config.json"
with open(config_path, 'r') as f:
    config = json.load(f)
config['device'] = 'cuda'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

# Update server checkpoint path
server_file = "/content/Evo-1/Evo_1/scripts/Evo1_server.py"
with open(server_file, 'r') as f:
    content = f.read()

# Fix checkpoint path
content = content.replace(
    'ckpt_dir = "/Users/tmprithvi/Code/EmbodiedLLM/vla-benchmark/models/Evo-1/checkpoints/libero"',
    'ckpt_dir = "/content/Evo-1/checkpoints/libero"'
)

with open(server_file, 'w') as f:
    f.write(content)

print("✅ Evo-1 server configured")

## 5. Install Miniconda and Setup LIBERO Environment

In [ ]:
%%bash
set -e

# Install Miniconda if not already installed
if [ ! -f /usr/local/bin/conda ]; then
    echo "📦 Installing Miniconda..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local -f
    rm Miniconda3-latest-Linux-x86_64.sh
    /usr/local/bin/conda init bash
    echo "✅ Miniconda installed"
else
    echo "✅ Conda already installed"
fi

# Accept conda Terms of Service
echo "📝 Accepting conda Terms of Service..."
/usr/local/bin/conda config --set allow_conda_downgrades true
yes | /usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
yes | /usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Create LIBERO environment
if [ ! -d /usr/local/envs/libero ]; then
    echo "📦 Creating LIBERO environment (Python 3.8.13)..."
    /usr/local/bin/conda create -p /usr/local/envs/libero python=3.8.13 -y
    echo "✅ Environment created"
else
    echo "✅ Environment already exists"
fi

## 6. Clone and Install LIBERO

In [ ]:
%%bash
set -e

# Clone LIBERO
mkdir -p /content/Evo-1/LIBERO_evaluation
cd /content/Evo-1/LIBERO_evaluation

if [ ! -d "LIBERO" ]; then
    echo "📦 Cloning LIBERO..."
    git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
    echo "✅ LIBERO cloned"
else
    echo "✅ LIBERO already exists"
fi

cd LIBERO

# Install dependencies in conda environment
echo "📦 Installing LIBERO dependencies..."
conda run -p /usr/local/envs/libero pip install -q -r requirements.txt
conda run -p /usr/local/envs/libero pip install -q torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113
conda run -p /usr/local/envs/libero pip install -q -e .
conda run -p /usr/local/envs/libero pip install -q websockets huggingface_hub

echo "✅ LIBERO installed"

## 7. Setup ngrok (Free - No Account)

Get your free ngrok authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken

Paste it below:

In [ ]:
# Set your ngrok authtoken here
NGROK_AUTH_TOKEN = "YOUR_TOKEN_HERE"  # Get free token from https://dashboard.ngrok.com/

from pyngrok import ngrok, conf
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok configured")

## 8. Run Everything (Simple Version)

In [ ]:
import subprocess
import time
from pyngrok import ngrok
import re
import os

# Ensure all server dependencies are installed
print("📦 Installing server dependencies...")
!pip install -q websockets opencv-python pillow torch torchvision transformers timm accelerate diffusers fvcore einops

# Kill any existing server
!fuser -k 9000/tcp 2>/dev/null || true
time.sleep(2)

# Start Evo-1 server in background
print("🚀 Starting Evo-1 server (takes 2-3 minutes to load model)...")
with open('/content/server.log', 'w') as log_file:
    server_process = subprocess.Popen(
        ["python", "Evo_1/scripts/Evo1_server.py"],
        cwd="/content/Evo-1",
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

print(f"✅ Server started (PID: {server_process.pid})")
print("📝 Logs: /content/server.log")

# Wait and check if server is actually running
for i in range(24):  # Check every 5 seconds for 2 minutes
    time.sleep(5)
    # Check if process is still alive
    if server_process.poll() is not None:
        print(f"\n❌ Server crashed! Exit code: {server_process.returncode}")
        print("\n📋 Last 50 lines of server log:")
        !tail -50 /content/server.log
        raise Exception("Server failed to start")
    
    # Check if port 9000 is listening
    result = !netstat -tuln 2>/dev/null | grep :9000 || true
    if result:
        print(f"\n✅ Server listening on port 9000 after {(i+1)*5} seconds")
        break
    
    if i % 3 == 0:
        print(f"⏳ Waiting... ({(i+1)*5}s elapsed)")

# Start ngrok tunnel
print("\n🌐 Starting ngrok tunnel...")
tunnel = ngrok.connect(9000, bind_tls=True)
public_url = tunnel.public_url.replace('https://', 'wss://')

print("\n" + "="*60)
print("✅ SERVER READY!")
print("="*60)
print(f"\n🌐 Public URL: {public_url}")
print("\n📝 For Mac client, update SERVER_URL to:")
print(f"   SERVER_URL = '{public_url}'")
print("\n" + "="*60)

# Check if client file exists (included in Evo-1 repo)
client_file = "/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py"
if os.path.exists(client_file):
    print("\n✅ Client file found")
    print("\nTo run client on Colab, execute next cell")
    print("Or run from your Mac with the URL above")
else:
    print("\n⚠️  Client file not found in Evo-1 repo!")

print("\n⚠️  Keep this cell running during evaluation!")

## 8a. Check Server Status (Run if connection fails)

If you see "connection refused" errors, run this cell to check server logs:

In [ ]:
!tail -100 /content/server.log

## 9. Run Client on Colab (Optional)

In [ ]:
# Update client to use localhost
client_file = "/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py"

if os.path.exists(client_file):
    with open(client_file, 'r') as f:
        client_content = f.read()
    
    # Update SERVER_URL to localhost
    import re
    client_content = re.sub(
        r'SERVER_URL = "[^"]*"',
        'SERVER_URL = "ws://localhost:9000"',
        client_content
    )
    
    with open(client_file, 'w') as f:
        f.write(client_content)
    
    print("✅ Client configured for localhost")
    print("🚀 Running LIBERO evaluation (2-3 hours)...\n")
    
    # Run client
    !cd /content/Evo-1/LIBERO_evaluation/LIBERO && conda run -p /usr/local/envs/libero python libero_client_4tasks.py
else:
    print("⚠️ Client file not found!")

## 10. Check Server Logs

In [ ]:
!tail -50 /content/server.log

## 11. Stop Server (When Done)

In [ ]:
# Stop server and ngrok
try:
    server_process.terminate()
    server_process.wait(timeout=5)
    print("✅ Server stopped")
except:
    !fuser -k 9000/tcp
    print("✅ Server killed")

ngrok.kill()
print("✅ ngrok stopped")